In [7]:
import sys
import os
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.gym_env import SnakeEnv
from core.env.enums import ObsType
from agents.q_learning import QLearningAgent

# 1. Increased episodes to allow the Q-table to fully converge
num_envs, total_episodes = 16, 250000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObsType.VECTOR_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
        )
        for i in range(num_envs)
    ]
)

# 2. Made epsilon decay reach its minimum at 70% of training, leaving the last 20% for pure exploitation fine-tuning
epsilon_decay = (0.01 / 1.0) ** (1 / (total_episodes * 0.7))

# 3. Increased gamma from 0.95 to 0.99.
# At 0.95, the agent only looks ~20 steps ahead (0.95^20 ≈ 0.35).
# At 0.99, it looks ~100 steps ahead, which is critical for making long maneuvers around obstacles.
# 4. Smoothed learning rate to 0.05 for late-stage stability
agent = QLearningAgent(
    state_dim=2048,
    action_dim=3,
    lr=0.05,
    gamma=0.99,
    epsilon_decay=epsilon_decay,
    seed=42,
)

training_logs, episode_rewards, completed = [], np.zeros(num_envs), 0
obs, infos = env.reset()

best_reward = -np.inf

with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:
        actions = [agent.act(o) for o in obs]
        next_obs, rewards, terms, truncs, next_infos = env.step(actions)

        for i, (o, a, r, no, term, trunc) in enumerate(
            zip(obs, actions, rewards, next_obs, terms, truncs)
        ):
            agent.update(o, a, r, no, term)
            episode_rewards[i] += r

            if term or trunc:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train()  # Decay epsilon per episode

                    reward_val = episode_rewards[i]
                    training_logs.append(
                        {
                            "episode": completed,
                            "reward": reward_val,
                            "epsilon": agent.epsilon,
                        }
                    )

                    if len(training_logs) >= 100:
                        recent_avg = np.mean(
                            [log["reward"] for log in training_logs[-100:]]
                        )
                        if recent_avg > best_reward:
                            best_reward = recent_avg
                            agent.save("q_learning_snake_best.pkl")

                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else reward_val
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (Shaped, last 100): {recent_avg:.2f} | Eps: {agent.epsilon:.3f} | Best Avg: {best_reward:.2f}"
                        )
                episode_rewards[i] = 0

        obs = next_obs
        infos = next_infos

env.close()

Parallel Training:   2%|▏         | 5052/250000 [00:10<08:44, 467.15it/s]

Ep 5000/250000 | Avg Reward (Shaped, last 100): 1.77 | Eps: 0.877 | Best Avg: 5.54


Parallel Training:   4%|▍         | 10091/250000 [00:21<08:38, 462.94it/s]

Ep 10000/250000 | Avg Reward (Shaped, last 100): 4.71 | Eps: 0.769 | Best Avg: 14.97


Parallel Training:   6%|▌         | 15083/250000 [00:33<08:57, 437.13it/s]

...

Ep 240000/250000 | Avg Reward (Shaped, last 100): 873.06 | Eps: 0.010 | Best Avg: 1003.36


Parallel Training:  98%|█████████▊| 245011/250000 [44:54<01:30, 55.15it/s]

Ep 245000/250000 | Avg Reward (Shaped, last 100): 796.81 | Eps: 0.010 | Best Avg: 1003.36


Parallel Training: 100%|██████████| 250000/250000 [46:26<00:00, 89.73it/s]

Ep 250000/250000 | Avg Reward (Shaped, last 100): 838.71 | Eps: 0.010 | Best Avg: 1003.36


In [8]:
agent.save("q_learning_snake.pkl")

In [9]:
from core.utils import save_metrics

save_metrics(training_logs, "q_learning_training_logs.csv")

In [10]:
from core.utils import evaluate_agent

evaluate_agent(agent)

Evaluating Agent: 100%|██████████| 100/100 [00:04<00:00, 20.16it/s]


Metric          | Average  | Max     
-----------------------------------
Rewards         | 1497.92  | 2921.55 
Apples          | 30.82    | 60.00   
Steps           | 694.17   | 1453.00 

Death Distribution:
 - self: 81 (81.0%)
 - wall: 19 (19.0%)



({'avg': 1497.9245000000074, 'max': 2921.549999999966},
 {'avg': 30.82, 'max': 60.0},
 {'avg': 694.17, 'max': 1453.0},
 {<DeathReason.SELF: 'self'>: 81, <DeathReason.WALL: 'wall'>: 19})